# 02 · Text-signal diagnostics
Are the transformer probabilities measuring what they claim? Correlation with the dictionary counts they were trained to imitate, dependence on text length, persistence across a firm's filings, and cross-factor correlation.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
from src.utils.config import path
T = path('results') / 'tables'
from src.data.risk import load_risk_scores, FACTORS
r = load_risk_scores(); r = r[r.words >= 1000]

In [2]:
rows = []
for f in FACTORS:
    rows.append({'factor': f, 'pearson_prob_count': r[f+'_prob'].corr(r[f+'_cnt']), 'spearman_prob_count': r[f+'_prob'].corr(r[f+'_cnt'], method='spearman'),
                 'spearman_prob_words': r[f+'_prob'].corr(r.words, method='spearman'), 'spearman_density_prob': (r[f+'_cnt']/r.words).corr(r[f+'_prob'], method='spearman')})
d = pd.DataFrame(rows).set_index('factor').round(3); d.to_csv(T/'text_diagnostics.csv'); d

,pearson_prob_count,spearman_prob_count,spearman_prob_words,spearman_density_prob
factor,,,,
accounting,0.588,0.616,0.491,0.598
finance,0.513,0.528,0.223,0.770
international,0.240,0.489,-0.033,0.697
legal,0.238,0.239,-0.047,0.621
management,0.112,0.156,0.064,0.327
marketing,0.108,0.230,-0.059,0.560
operations,-0.103,-0.070,-0.382,0.699
technology,0.029,0.150,-0.035,0.613


Low probability–count correlations and a *negative* relation with length mean the probability behaves like a share of risk language, not an amount. Ranking within each month's cross-section (the design used throughout) removes the secular length trend.

In [3]:
s = r.sort_values(['cik','filing_date']); g = s.groupby('cik')
persist = {f: s[f+'_prob'].corr(g[f+'_prob'].shift(1)) for f in FACTORS}
pd.Series(persist, name='autocorr_consecutive_filings').round(3)

accounting       0.798
finance          0.904
international    0.714
legal            0.838
management       0.563
marketing        0.792
operations       0.810
technology       0.796
Name: autocorr_consecutive_filings, dtype: float64

In [4]:
r[[f+'_prob' for f in FACTORS]].corr().round(2)

,accounting_prob,finance_prob,international_prob,legal_prob,management_prob,marketing_prob,operations_prob,technology_prob
accounting_prob,1.00,0.46,-0.10,-0.44,0.33,-0.16,-0.34,-0.12
finance_prob,0.46,1.00,-0.28,-0.45,0.18,-0.44,-0.29,-0.39
international_prob,-0.10,-0.28,1.00,0.08,0.16,0.32,0.33,0.35
legal_prob,-0.44,-0.45,0.08,1.00,-0.16,0.12,-0.14,0.09
management_prob,0.33,0.18,0.16,-0.16,1.00,0.16,0.09,0.19
marketing_prob,-0.16,-0.44,0.32,0.12,0.16,1.00,0.16,0.40
operations_prob,-0.34,-0.29,0.33,-0.14,0.09,0.16,1.00,0.20
technology_prob,-0.12,-0.39,0.35,0.09,0.19,0.40,0.20,1.00
